# k-NN Classifier on VGG-16 Block4 Pool Features

This notebook trains a k-nearest-neighbors baseline on the precomputed feature extraction artefacts from Alena's archive.

In [ ]:
from pathlib import Path
import json
import os

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
import wandb
from wandb.errors import CommError
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

config_params = {
    "model_type": "KNN",
    "feature_source": "vgg16_block4pool_alena_archive",
    "feature_dimensions": 512,
    "n_neighbors": 5,
    "weights": "distance",
    "metric": "minkowski",
    "p": 2,
    "implementation": "exact_batched_torch",
    "batch_size": 64,
    "test_size": 0.20,
    "random_state": 42,
}

os.environ.setdefault("WANDB_NOTEBOOK_NAME", "notebooks/04_knn.ipynb")

wandb_mode = os.environ.get("WANDB_MODE")
if not wandb_mode and not os.environ.get("WANDB_API_KEY") and not Path.home().joinpath("_netrc").exists():
    wandb_mode = "offline"
    os.environ["WANDB_MODE"] = wandb_mode
    print("WANDB_API_KEY/_netrc not found. Falling back to offline wandb mode.")

wandb_init_kwargs = {
    "project": "odysseus",
    "name": "vgg16_block4pool_knn_distance",
    "job_type": "modeling",
    "config": config_params,
    "mode": wandb_mode,
    "save_code": False,
}

try:
    wandb.init(**wandb_init_kwargs)
except CommError as exc:
    if wandb_mode == "offline":
        raise
    print(f"wandb login unavailable ({exc}). Falling back to offline mode.")
    os.environ["WANDB_MODE"] = "offline"
    wandb_mode = "offline"
    wandb.init(**{**wandb_init_kwargs, "mode": wandb_mode})
config = wandb.config

In [ ]:
def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for path in [start, *start.parents]:
        if (path / "data" / "extracted_features_alena").exists():
            return path
    raise FileNotFoundError(
        "Could not find data/extracted_features_alena from the current working directory."
    )


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "extracted_features_alena"
print(f"Project root: {PROJECT_ROOT}")
print(f"Feature directory: {DATA_DIR}")


X = np.load(DATA_DIR / "features_block4pool.npy")
y = np.load(DATA_DIR / "labels_block4pool.npy")

with open(DATA_DIR / "class_indices.json", "r", encoding="utf-8") as f:
    class_to_idx = json.load(f)

idx_to_class = {idx: class_name for class_name, idx in class_to_idx.items()}
class_names = [idx_to_class[idx] for idx in sorted(idx_to_class)]

print(f"Loaded X: {X.shape}, y: {y.shape}")
print(f"Classes: {len(class_names)}")
print(class_names)

assert X.ndim == 2, f"Expected 2D feature matrix, got shape {X.shape}"
assert y.ndim == 1, f"Expected 1D labels, got shape {y.shape}"
assert len(X) == len(y), "Feature and label counts do not match"
assert X.shape[1] == config.feature_dimensions, (
    f"Expected {config.feature_dimensions} features, got {X.shape[1]}"
)

In [ ]:
unique_labels, label_counts = np.unique(y, return_counts=True)
distribution = {
    idx_to_class[int(label)]: int(count)
    for label, count in zip(unique_labels, label_counts)
}

plt.figure(figsize=(14, 5))
sns.barplot(x=list(distribution.keys()), y=list(distribution.values()), color="steelblue")
plt.title("Class Distribution - VGG-16 Block4 Pool Features")
plt.xlabel("Letter")
plt.ylabel("Samples")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig("knn_class_distribution.png", dpi=300)
wandb.log({"class_distribution": wandb.Image("knn_class_distribution.png")})
plt.show()

distribution

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=config.test_size,
    random_state=config.random_state,
    stratify=y,
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train: {X_train_scaled.shape}, Test: {X_test_scaled.shape}")

In [ ]:
def predict_exact_weighted_knn(X_train, y_train, X_test, n_classes, k=5, batch_size=64):
    train = torch.from_numpy(X_train.astype(np.float32, copy=False))
    test = torch.from_numpy(X_test.astype(np.float32, copy=False))
    y_train_t = torch.from_numpy(y_train.astype(np.int64, copy=False))

    train_t = train.T.contiguous()
    train_norm = (train * train).sum(dim=1)
    y_pred = np.empty(len(X_test), dtype=y_train.dtype)

    for start_idx in range(0, len(test), batch_size):
        end_idx = min(start_idx + batch_size, len(test))
        batch = test[start_idx:end_idx]
        dists = (batch * batch).sum(dim=1, keepdim=True) + train_norm[None, :] - 2 * batch.matmul(train_t)
        nn_dists, nn_idx = torch.topk(dists, k=k, largest=False, dim=1)
        nn_labels = y_train_t[nn_idx]
        weights = 1 / (torch.sqrt(torch.clamp(nn_dists, min=0)) + 1e-12)

        for row_offset, (labels, row_weights) in enumerate(zip(nn_labels, weights)):
            scores = torch.zeros(n_classes, dtype=torch.float64)
            scores.scatter_add_(0, labels.long(), row_weights.double())
            y_pred[start_idx + row_offset] = int(scores.argmax())

        if end_idx == len(test) or end_idx % (batch_size * 10) == 0:
            print(f"Predicted {end_idx}/{len(test)}")

    return y_pred


print("Generating exact batched k-NN predictions...")
y_pred = predict_exact_weighted_knn(
    X_train_scaled,
    y_train,
    X_test_scaled,
    n_classes=len(class_names),
    k=config.n_neighbors,
    batch_size=config.batch_size,
)

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average="macro")
weighted_f1 = f1_score(y_test, y_pred, average="weighted")

print(f"Accuracy: {accuracy:.4f}")
print(f"Macro-F1: {macro_f1:.4f}")
print(f"Weighted-F1: {weighted_f1:.4f}")

wandb.run.summary["final_accuracy"] = accuracy
wandb.run.summary["final_macro_f1"] = macro_f1
wandb.run.summary["final_weighted_f1"] = weighted_f1

report = classification_report(
    y_test,
    y_pred,
    target_names=class_names,
    digits=4,
)

print("\nClassification Report (per class):")
print(report)

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=sorted(idx_to_class), normalize="true")

plt.figure(figsize=(16, 14))
sns.heatmap(
    cm,
    annot=True,
    fmt=".2f",
    cmap="viridis",
    cbar=True,
    xticklabels=class_names,
    yticklabels=class_names,
    annot_kws={"size": 7},
)
plt.title(f"Normalized Confusion Matrix - VGG-16 Block4 Pool + k-NN (Acc: {accuracy:.2f})")
plt.xlabel("Predicted Letter")
plt.ylabel("True Letter")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()

plt.savefig("knn_confusion_matrix.png", dpi=300)
wandb.log({"confusion_matrix_annotated": wandb.Image("knn_confusion_matrix.png")})
plt.show()

In [ ]:
wandb.finish()